# Mission Control AI - GS 2026.1
**Prompt and Artificial Intelligence - FIAP**

Sistema inteligente de monitoramento de uma missao espacial experimental.
Gera dados simulados de telemetria, aplica logica de alertas e decisao,
e usa o modelo **Llama 3.2 (1B) via Ollama** para analisar a situacao e
recomendar acoes com contexto da missao.

> Execute as celulas **em ordem, de cima para baixo**.

## 1. Configuracao do ambiente (Ollama)

In [ ]:
# Instala o Ollama no Colab
!curl -fsSL https://ollama.com/install.sh | sh

# Inicia o servidor Ollama em background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Servidor Ollama iniciado.")

In [ ]:
# Baixa o modelo Llama 3.2 1B (leve, roda direto no Colab)
!ollama pull llama3.2:1b

# Instala a biblioteca Python do Ollama
!pip install ollama -q
print("Modelo e biblioteca prontos.")

## 2. Imports e configuracao da missao

In [ ]:
import ollama
import random
import time
from datetime import datetime

MODELO_IA = "llama3.2:1b"

# Limites operacionais (thresholds) de cada parametro monitorado
LIMITES = {
    "temperatura": {"min": -10, "max": 70, "critico_max": 85},  # graus C
    "energia":     {"min": 20, "critico_min": 10},               # % de bateria
    "sinal":       {"min": 40},                                  # % qualidade do sinal
}

# Modulos da estacao/nave monitorados
MODULOS = ["Comando", "Suporte de Vida", "Laboratorio", "Energia", "Comunicacao"]

print("Configuracao carregada. Modulos monitorados:", MODULOS)

## 3. Simulacao de dados da missao
Gera dados simulados de telemetria. O parametro `cenario` permite forcar uma
situacao **normal** ou **critica** (util para demonstrar os alertas no video).

In [ ]:
def gerar_telemetria(cenario="aleatorio"):
    """Gera dados simulados de telemetria da missao espacial."""
    if cenario == "critico":
        temp    = random.randint(86, 110)
        energia = random.randint(3, 18)
        sinal   = random.randint(5, 35)
    elif cenario == "normal":
        temp    = random.randint(15, 45)
        energia = random.randint(60, 100)
        sinal   = random.randint(70, 100)
    else:  # aleatorio
        temp    = random.randint(-15, 110)
        energia = random.randint(3, 100)
        sinal   = random.randint(5, 100)

    return {
        "timestamp":     datetime.now().strftime("%H:%M:%S"),
        "modulo":        random.choice(MODULOS),
        "temperatura":   temp,                          # graus C
        "energia":       energia,                       # % bateria
        "geracao_solar": round(random.uniform(0, 3.5), 2),  # kW
        "sinal":         sinal,                         # % qualidade
    }

# Teste rapido
print(gerar_telemetria("normal"))

## 4. Logica de alertas e tomada de decisao
Aqui esta o "cerebro" baseado em regras: avalia cada parametro contra os
limites, gera **alertas automaticos** e define **acoes automatizadas**
para situacoes criticas.

In [ ]:
def avaliar_telemetria(dados):
    """Aplica logica de decisao e gera alertas a partir dos limites."""
    alertas, acoes = [], []
    severidade = 0  # 0 = NORMAL, 1 = ATENCAO, 2 = CRITICO

    # --- Temperatura ---
    t = dados["temperatura"]
    if t >= LIMITES["temperatura"]["critico_max"]:
        alertas.append(f"[CRITICO] Temperatura {t}C acima do limite seguro")
        acoes.append(f"Acionar resfriamento de emergencia no modulo {dados['modulo']}")
        severidade = max(severidade, 2)
    elif t > LIMITES["temperatura"]["max"] or t < LIMITES["temperatura"]["min"]:
        alertas.append(f"[ATENCAO] Temperatura {t}C fora da faixa ideal")
        acoes.append("Reforcar monitoramento do controle termico")
        severidade = max(severidade, 1)

    # --- Energia ---
    e = dados["energia"]
    if e <= LIMITES["energia"]["critico_min"]:
        alertas.append(f"[CRITICO] Bateria em {e}% - risco de desligamento")
        acoes.append("Ativar modo de sobrevivencia e cortar cargas nao essenciais")
        severidade = max(severidade, 2)
    elif e < LIMITES["energia"]["min"]:
        alertas.append(f"[ATENCAO] Bateria baixa ({e}%)")
        acoes.append("Ativar modo de economia de energia")
        severidade = max(severidade, 1)

    # --- Comunicacao ---
    s = dados["sinal"]
    if s < LIMITES["sinal"]["min"]:
        alertas.append(f"[ATENCAO] Sinal de comunicacao fraco ({s}%)")
        acoes.append("Reorientar antena e priorizar reenvio de pacotes")
        severidade = max(severidade, 1)

    nivel = {0: "NORMAL", 1: "ATENCAO", 2: "CRITICO"}[severidade]
    return {"nivel": nivel, "alertas": alertas, "acoes": acoes}

# Teste rapido com um cenario critico
print(avaliar_telemetria(gerar_telemetria("critico")))

## 5. Inteligencia Artificial (Llama via Ollama)
O modelo de linguagem recebe a telemetria + os alertas e devolve uma
**analise em linguagem natural com contexto da missao espacial**, priorizando
o problema mais urgente e recomendando uma acao. Ha um *fallback* por regras
para garantir que o sistema nunca quebre se o modelo demorar a responder.

In [ ]:
SYSTEM_PROMPT = (
    "Voce e o nucleo de inteligencia da Mission Control AI, responsavel por "
    "monitorar uma missao espacial experimental. Voce recebe dados de telemetria "
    "(temperatura, energia, sinal de comunicacao) e os alertas gerados pelo sistema. "
    "Sua funcao e: (1) avaliar o estado geral da missao em linguagem clara, "
    "(2) apontar o problema mais urgente e (3) recomendar uma acao objetiva. "
    "Responda em portugues, em no maximo 4 frases curtas. Seja direto e tecnico."
)

def analisar_com_ia(dados, avaliacao):
    """Usa o modelo Llama para analisar a missao com contexto espacial."""
    contexto = (
        f"Modulo: {dados['modulo']}\n"
        f"Temperatura: {dados['temperatura']} C\n"
        f"Energia (bateria): {dados['energia']}%\n"
        f"Geracao solar: {dados['geracao_solar']} kW\n"
        f"Sinal de comunicacao: {dados['sinal']}%\n"
        f"Nivel de alerta do sistema: {avaliacao['nivel']}\n"
        f"Alertas ativos: {avaliacao['alertas'] or 'nenhum'}\n"
        "Analise a situacao da missao e recomende a acao prioritaria."
    )
    try:
        resposta = ollama.chat(
            model=MODELO_IA,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": contexto},
            ],
        )
        return resposta["message"]["content"].strip()
    except Exception:
        # Fallback por regras (resposta automatizada de seguranca)
        if avaliacao["nivel"] == "CRITICO":
            return "Situacao critica. Executar imediatamente as acoes automaticas recomendadas."
        if avaliacao["nivel"] == "ATENCAO":
            return "Parametros em atencao. Monitoramento reforcado recomendado."
        return "Todos os parametros dentro da faixa nominal. Missao estavel."

## 6. Painel de visualizacao (saida organizada)

In [ ]:
def exibir_painel(dados, avaliacao, analise_ia):
    """Exibe os dados da missao de forma organizada no terminal/Colab."""
    icone = {"NORMAL": "[ OK ]", "ATENCAO": "[ ! ]", "CRITICO": "[ XX ]"}
    print("=" * 58)
    print("            MISSION CONTROL AI - STATUS ATUAL")
    print("=" * 58)
    print(f" Horario       : {dados['timestamp']}")
    print(f" Modulo        : {dados['modulo']}")
    print("-" * 58)
    print(f" Temperatura   : {dados['temperatura']} C")
    print(f" Energia       : {dados['energia']} %")
    print(f" Geracao solar : {dados['geracao_solar']} kW")
    print(f" Comunicacao   : {dados['sinal']} %")
    print("-" * 58)
    print(f" NIVEL GERAL   : {icone[avaliacao['nivel']]} {avaliacao['nivel']}")
    if avaliacao["alertas"]:
        print(" ALERTAS:")
        for a in avaliacao["alertas"]:
            print(f"   - {a}")
        print(" ACOES AUTOMATICAS:")
        for ac in avaliacao["acoes"]:
            print(f"   > {ac}")
    else:
        print(" ALERTAS       : Nenhum. Todos os sistemas nominais.")
    print("-" * 58)
    print(" ANALISE DA IA (Llama 3.2):")
    for linha in analise_ia.split("\n"):
        print(f"   {linha}")
    print("=" * 58)
    print()

## 7. Monitoramento automatico da missao
Roda varios ciclos: gera dados, avalia, chama a IA e exibe o painel.
Este e o coracao da demonstracao.

In [ ]:
def executar_monitoramento(ciclos=4, intervalo=1, cenario="aleatorio"):
    print(">>> Iniciando monitoramento da missao espacial...\n")
    time.sleep(1)
    for i in range(ciclos):
        print(f"--- Ciclo {i + 1}/{ciclos} ---")
        dados     = gerar_telemetria(cenario)
        avaliacao = avaliar_telemetria(dados)
        analise   = analisar_com_ia(dados, avaliacao)
        exibir_painel(dados, avaliacao, analise)
        time.sleep(intervalo)
    print(">>> Monitoramento encerrado.")

# Executa o ciclo de monitoramento (dados aleatorios)
executar_monitoramento(ciclos=4, intervalo=1)

## 8. Demonstracao de um cenario CRITICO
Forca uma emergencia para mostrar os alertas, as acoes automaticas e a
resposta da IA. **Bom para o video de demonstracao** e para o print de alerta.

In [ ]:
print(">>> SIMULANDO CENARIO CRITICO <<<\n")
dados     = gerar_telemetria("critico")
avaliacao = avaliar_telemetria(dados)
analise   = analisar_com_ia(dados, avaliacao)
exibir_painel(dados, avaliacao, analise)

## 9. Previsao (analise de tendencia)
Alem da analise da IA, o sistema faz uma **previsao simples**: a partir da
tendencia de descarga da bateria, estima em quantos ciclos ela se esgota.
Isso atende ao criterio de "exibir previsoes a partir dos dados".

In [ ]:
def prever_autonomia(historico_energia):
    """Estima quantos ciclos ate a bateria zerar (tendencia linear simples)."""
    if len(historico_energia) < 2:
        return None
    quedas = [historico_energia[i] - historico_energia[i + 1]
              for i in range(len(historico_energia) - 1)]
    taxa = sum(quedas) / len(quedas)
    if taxa <= 0:
        return None  # bateria estavel ou carregando
    return round(historico_energia[-1] / taxa, 1)

def executar_previsao_energia(ciclos=6):
    print(">>> MODO PREVISAO - Monitorando descarga da bateria <<<\n")
    energia, historico = random.randint(70, 90), []
    for i in range(ciclos):
        energia = max(0, energia - random.randint(8, 16))   # descarga progressiva
        historico.append(energia)
        dados = gerar_telemetria("normal")
        dados["energia"] = energia
        avaliacao = avaliar_telemetria(dados)
        previsao  = prever_autonomia(historico)
        linha = f"Ciclo {i + 1}: Bateria {energia}%"
        if previsao is not None:
            linha += f"  | Previsao: ~{previsao} ciclos ate esgotar"
        print(linha)
        if avaliacao["nivel"] != "NORMAL":
            print(f"   IA: {analisar_com_ia(dados, avaliacao)}")
        time.sleep(0.5)
    print("\n>>> Previsao concluida.")

executar_previsao_energia(ciclos=6)

## 10. (Opcional) Modo conversacional - Chatbot
Interface de chat com a Mission Control AI. Comandos: `status`, `critico`,
ou qualquer pergunta livre. Digite `sair` para encerrar.
*(Opcional - nao vale pontos extras, mas e otimo para o video.)*

In [ ]:
def chatbot_missao():
    print("Mission Control AI - modo conversacional")
    print("Comandos: 'status', 'critico', ou pergunta livre. Digite 'sair' para encerrar.\n")
    estado = gerar_telemetria("normal")
    while True:
        comando = input("Voce > ").strip().lower()
        if comando in ("sair", "exit", "quit"):
            print("Encerrando Mission Control AI.")
            break
        if comando in ("status", "critico"):
            estado = gerar_telemetria("critico" if comando == "critico" else "aleatorio")
            av = avaliar_telemetria(estado)
            exibir_painel(estado, av, analisar_com_ia(estado, av))
        else:
            try:
                r = ollama.chat(model=MODELO_IA, messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": f"Dados atuais: {estado}. Pergunta: {comando}"},
                ])
                print("IA >", r["message"]["content"].strip(), "\n")
            except Exception:
                print("IA > Nao consegui processar agora. Tente 'status'.\n")

# Descomente a linha abaixo para usar o chatbot:
# chatbot_missao()